In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


import pandas as pd
import numpy as np
import tensorflow as tfs
from tensorflow import keras
import matplotlib.pyplot as plt
from keras.layers import Dense, Conv2D, MaxPool2D,Flatten, Input,Dropout
import seaborn as sns
np.random.seed(42)
tfs.random.set_seed(42)

# 1.Introducción
Presentar el problema de clasificación con la descripción del conjunto de datos indicando la variante elegida y el objetivo de implementar arquitectura CNN para el problema. Explicar porque es pertinente comparar este modelo con el MLP anterior.




#   2.Preparación del conjunto de datos
Describir proceso de preparación de datos.

En esta sección tambien va:

*   Carga del dataset
*   Redimensionamiento cuando corresponda
*   Normalización
*   Conservación formato matricial de las imagenes
*   Partición en entrenamiento, validación y prueba
*   Visualización de ejemplos representativos

Cuando se reutilice el trabajo de la Ev1, se debera explicar las modificaciones necesarias para adaptar el dataset al nuevo modelo.



### Seccion cargar dataset signLanguage

In [ ]:
# importar dataset signLanguage, separar datos para train y test.

train = pd.read_csv('../data/sign_mnist_train.csv')
test = pd.read_csv('../data/sign_mnist_test.csv')

In [ ]:
print(f'''
Datos entrenamiento
dimension: {train.shape}
cantidad de datos : {train.shape[0]} imagenes
dimension de cada imagen: {train.shape[1]} pixeles por imagen''')


In [ ]:
print(f'''
Datos entrenamiento
dimension: {test.shape}
cantidad de datos : {test.shape[0]} imagenes
dimension de cada imagen: {test.shape[1]} pixeles por imagen''')

In [ ]:
train.head(5)

In [ ]:
test.head(5)

In [ ]:
print(train.info())

In [ ]:
print(test.info())

In [ ]:
# Cantidad de pixeles de la imagen y su dimension
print(f'''Cantidad de pixeles de la imagen: {train.shape[1]-1}
dimension: {np.sqrt(train.shape[1]-1)}
dimension imagen: {int(np.sqrt(train.shape[1]-1))} X {int(np.sqrt(train.shape[1]-1))}''')

In [ ]:
clases_in_train = train['label'].sort_values(ascending=False).unique()
clases_in_test = test['label'].sort_values(ascending=False).unique()
print(f'''
clase train:{clases_in_train}
clase test: {clases_in_test}''')


In [ ]:
# Drop para la columna label en train
features_train = train.drop(columns='label')
labels_train = train['label']

# Drop para la columna label en test
features_test = test.drop(columns='label')
labels_test = test['label']

In [ ]:
# normalizar los datos de los imagenes
def normalizar(imagenes,etiquetas):
    """conversion de valores de los valores de los
    pixeles de 255 a 0-1.
    Args:
        imagenes (_type_): se transforman los valores de pixeles a binario.
    """
    imagenes = tfs.cast(imagenes,tfs.float32)
    imagenes/=255
    return imagenes,etiquetas

In [ ]:
# normalizacion para train and test
normalizar_train, label_train = normalizar(features_train,labels_train)
normalizar_test, label_test = normalizar(features_test,labels_test)

In [ ]:
class_distribution = labels_train.value_counts(ascending=False)
diference_class = class_distribution.max() - class_distribution.min()

In [ ]:
train_counts= {}
for number in clases_in_train:
    train_counts[number] = label_train.value_counts()[number]

test_count={}
for number in clases_in_test:
    test_count[number] = label_test.value_counts()[number]

In [ ]:
#  analisis de etiquetas para train y test
diference_class = pd.DataFrame({
    'clases':clases_in_train,
    'train_counts':[train_counts[c] for c in clases_in_train],
    'test_count':[test_count[c] for c in clases_in_test]
})
diference_class.sort_values(by='clases')

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(
    diference_class['clases'],
    diference_class['train_counts']
)
plt.title("Imagénes de entrenamiento por clase", fontsize=14, fontweight="bold")
plt.xlabel("Clase", fontsize=12, fontweight="bold")
plt.ylabel("Número de imágenes", fontsize=12, fontweight="bold")
plt.xticks(rotation=45)
plt.show()

In [ ]:
diference_class_long = diference_class.melt(id_vars='clases',value_vars=['train_counts','test_count'])
diference_class_long['data'] = diference_class_long['variable'].map({
    'train_counts':'Entrenamiento',
    'test_count':'Prueba'
})
sns.barplot(data=diference_class_long,x='clases',y='value',hue='data')
plt.ylabel('Cantidad')
plt.xlabel('Clases')
plt.title('Distribucion de clases: entrenamiento vs prueba')

In [ ]:
# normalizacion de espacio dimensional posteriormente quedan en un espacio de tensor 1 vector.
data_train = tfs.data.Dataset.from_tensor_slices((normalizar_train,label_train))
data_test = tfs.data.Dataset.from_tensor_slices((normalizar_test,label_test))

In [ ]:
# Aquí usamos el reshape para darle a TensorFlow la estructura de las imágenes.
# Esto se realiza porque inicialmente cada imagen está representada como
# un vector de 784 píxeles. Como sabemos que 784 = 28 x 28, reorganizamos
# esos valores para que TensorFlow interprete cada imagen como una matriz
# de 28 x 28 píxeles.
x_train = tfs.reshape(normalizar_train,(-1,28,28,1))
x_test  = tfs.reshape(normalizar_test,(-1,28,28,1))

In [ ]:
# diccionario que contiene las letras segun su clase.
abecedario = {
    'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4,
    'F': 5, 'G': 6, 'H': 7, 'J': 8,
    'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14,
    'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19,
    'U': 20, 'V': 21, 'W': 22, 'X': 23
}

f, ax = plt.subplots(2,5) # mostramos 10 imagenes su dimension es de 2x5, es decir 2 filas x 5 columnas.
f.set_size_inches(10,10) # modificar tamano a pulgadas.
k = 0 # k es nuestro indice para decir que imagen debe mostrar de x_train.
for i in range(2):
    for j in range(5):
# como x_train[k] pide una imagen y nuestra estructura es de 4d, aqui reshape dice que solo quiere la dimension de la imagen que es de 28x28.
# y temporalmente se quita el canal 1.
        letras = {valor:clave for clave,valor in abecedario.items()}
        ax[i,j].imshow(tfs.reshape(x_train[k],(28,28)), cmap='gray')
        ax[i,j].set_title(f"Clase: {letras[label_train[k]]}")
        letras

        k+=1 # aumentamos en indice de k en 1 hasta llegar a 10 que es el total de imagenes que queremos.
    plt.tight_layout()

In [ ]:
# Remapeo de etiquetas, se aplica para ordenar las clases, ya que no existe la clase j(9) y z(25).
print(np.unique(label_train))
print(np.unique(label_test))
labels = [i for i in range(25) if i != 9]
mapping= {label: i for i, label in enumerate(labels)}
label_train=np.array([mapping[label]for label in label_train])
label_test=np.array([mapping[label]for label in label_test])

# 3.Diseño de la CNN
Implementar arquitectura CNN adecuada para resolver.

Se debe describir:
*   Cantidad de capas convolucionales
*   Tamaño de los filtros
*   Funciones de activación
*   Capas de pooling
*   Capas densas finales
*   Función de perdida
*   Optimizador
*   Hiperparametros utilizados

**Justificar cada decisión.**

In [ ]:
# Red CNN ,1 conv, 1maxpool
red_cnn = keras.Sequential(name='cnn')
red_cnn.add(Input(shape=(28,28,1),name='Input'))
red_cnn.add(Conv2D(64,(3,3),activation='relu',name='conv_1'))
red_cnn.add(MaxPool2D((2,2),name='maxpool_1'))
red_cnn.add(Flatten(name='flatten_1'))
red_cnn.add(Dense(128, activation='relu',name='Dense_1'))
red_cnn.add(Dense(24,activation ='softmax',name='salida'))
red_cnn.summary()

In [ ]:
# cnn2, 2 conv2d , 2 maxpool
# cnn2 = keras.Sequential(name='cnn2')
# cnn2.add(Input(shape=(28,28,3),name='Entrada'))
# cnn2.add(Conv2D(64,(3,3),activation='relu',name='conv2D_1'))
# cnn2.add(MaxPool2D((2,2),name='MaxPool2D_1'))
# cnn2.add(Conv2D(14,(3,3),activation='relu',name='conv2D_2'))
# cnn2.add(MaxPool2D((2,2),name='MaxPool2D_2'))
# cnn2.add(Flatten(name='flatten_1'))
# cnn2.add(Dropout(0.5)) # al 50%
# cnn2.add(Dense(350, activation='relu',name='dense_1'))
# cnn2.add(Dense(24,activation='softmax',name='salida'))
# cnn2.summary()


In [ ]:
# cnn3 = keras.Sequential(name='cnn3')
# cnn3.add(Input(shape=(28,28,3),name='Entrada'))
# cnn3.add(Conv2D(64,(3,3),activation='relu',name='conv2D_1'))
# cnn3.add(MaxPool2D((2,2),name='MaxPool2D_1'))
# cnn3.add(Conv2D(14,(3,3),activation='relu',name='conv2D_2'))
# cnn3.add(MaxPool2D((2,2),name='MaxPool2D_2'))
# cnn3.add(Flatten(name='flatten_1'))
# cnn3.add(Dense(350, activation='relu',name='dense_1'))
# cnn3.add(Dense(24,activation='softmax',name='salida'))
# cnn3.summary()

# 4.Entrenamiento del modelo
Entrenar la red registrando el comportamiento del proceso de aprendizaje. Se espera que se incorporen curvas de entrenamiento y validación, en la que se debe de analizar la evolución de la perdida y la precisión durantes las distintas epocas.

In [ ]:
# semilla y numero de epocas en 25.
red_cnn.compile(
    optimizer = 'adam',
    loss= 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)
hist = red_cnn.fit(
    # datos de train y test
    batch_size = 64,
    epochs=25,
    validation_data =()
)

# 5.Evaluación modelo
Evaluar desempeño utilizando:
*   Accuracy
*   Precision
*   Recall
*   F1-score
*   Matriz de confusión

Se deben de interpretar las metricas considerando el comportamiento observado durante el entrenamiento.

# 6.Comparación entre MLP y CNN
Comparar ambos modelos, analizando como minimo:
*   Desempeño alcanzado
*   Cantidad de parametros
*   Tiempo de entrenamiento
*   Capacidad de generalización
*   Ventajas y desventajas de cada arquitectura

**Sustentar mediante tablas y graficos.**

# 7.Visualización de mapas de caracteristicas y activación
Seleccionar algunas imagenes del dataset de prueba y visualizar Features Maps generados por distintas capas convolucionales.
Analizar la evolución de la informacion a medida que aumenta la profundidad de la red y discutir el aporte de las capas convolucionales para la extraccion automatica de caracteristicas.
Visualizar el mapa de activacion de cada clase.